# Multi-Prompt Optimization with LangMem

## What is Multi-Prompt Optimization?

Most real-world agents are **pipelines**: a sequence of LLM-powered steps, each with its own system prompt. For example:

```
User Query → [Classifier] → [Researcher] → [Writer] → Final Answer
```

Each step has its own prompt. When something goes wrong in the final output, you face a classic problem: **which prompt is to blame?**

`create_multi_prompt_optimizer` solves this with **credit assignment** — it reads the full conversation trajectory and figures out *which* prompt in the pipeline caused the failure, then surgically updates that prompt (and leaves the others alone).

---

## Key Difference from `create_prompt_optimizer`

| | `create_prompt_optimizer` | `create_multi_prompt_optimizer` |
|---|---|---|
| **Prompts** | One system prompt | List of named `Prompt` dicts |
| **Credit assignment** | N/A | Automatically blames the right prompt |
| **Input key** | `"prompt"` | `"prompts"` |
| **Output** | Single improved string | List of improved `Prompt` dicts |
| **Best for** | Single-step agents | Multi-step pipelines / agent teams |

---

## The Credit Assignment Problem

Imagine this pipeline failure:

```
User: "What's the refund policy?"
Classifier prompt → (incorrectly tags it as a "technical" issue)
Responder prompt → (correctly explains a technical issue — but wrong topic!)
User: "That's not what I asked about"
```

Who's at fault? The **Classifier** — the Responder did its job fine given what it received. A naive approach would update *both* prompts. `create_multi_prompt_optimizer` is smart enough to only update the Classifier.

This matters because **updating a prompt that wasn't broken can make things worse**.

## Setup

In [1]:
from dotenv import load_dotenv
load_dotenv()

from langmem import create_multi_prompt_optimizer

---

## Our Scenario: A Customer Support Pipeline

We'll simulate a two-stage customer support agent:

```
User Message
    │
    ▼
┌─────────────┐
│  Classifier  │  ← Prompt 1: decides if issue is "billing", "technical", or "general"
└─────┬───────┘
      │  (category label)
      ▼
┌─────────────┐
│  Responder  │  ← Prompt 2: crafts a reply tuned to the category
└─────────────┘
      │
      ▼
Final Answer
```

Both prompts start simple and under-specified — we'll use the optimizer to improve them based on real conversation outcomes.

In [2]:
# Step 1: Define the starting prompts for each stage of the pipeline.
# These are intentionally bare — just like you'd write on day one of a new project.
# The optimizer will flesh them out based on real failures.

prompts = [
    {
        "name": "classifier",
        "prompt": "You classify customer support messages into categories.",
    },
    {
        "name": "responder",
        "prompt": "You respond to customer support messages.",
    },
]

# Print what we're starting with
for p in prompts:
    print(f"[{p['name']}]")
    print(f"  {p['prompt']}")
    print()

[classifier]
  You classify customer support messages into categories.

[responder]
  You respond to customer support messages.



---

## Step 2: Define Trajectories

Each trajectory is a `(conversation, annotation)` pair — the same format as `create_prompt_optimizer`.

For a multi-agent pipeline, the conversation shows the **full sequence** of messages produced by all agents. The optimizer reads this holistically and attributes failures to specific prompts.

We'll define **three trajectories** that showcase different failure types:

| Trajectory | What failed | Which prompt is to blame |
|---|---|---|
| #1 — Billing misclassified | Classifier tagged a billing question as "general" | **classifier** |
| #2 — Cold tone in response | Responder gave a correct but robotic reply | **responder** |
| #3 — Happy path | Everything worked | Neither (positive signal) |

In [3]:
# ────────────────────────────────────────────────────────────────────────────
# Trajectory 1: Classifier is to blame
#
# The user asks a billing question. The classifier wrongly tags it as "general".
# The responder then gives a generic answer (reasonable given the wrong tag).
# The user has to push back explicitly.
#
# Credit assignment target: CLASSIFIER
# ────────────────────────────────────────────────────────────────────────────
trajectory_1 = (
    [
        {"role": "user",      "content": "I was charged twice for my subscription this month."},
        # Classifier's output (simulated as an assistant turn showing its decision)
        {"role": "assistant", "content": "[Category: general] I understand you have a concern."},
        # Responder's output — gave a vague reply because it received "general" category
        {"role": "assistant", "content": "Thank you for reaching out! Is there anything else I can help you with today?"},
        # User explicitly signals the response missed the mark
        {"role": "user",      "content": "That didn't help at all. I need a refund for the duplicate charge."},
    ],
    # Annotation: the revised response the system SHOULD have given.
    # This makes it clear a billing-specific response was expected.
    {
        "revised": (
            "[Category: billing] I can see you were charged twice for your subscription. "
            "I sincerely apologize for this error. I'll process a refund for the duplicate "
            "charge immediately — it will appear on your statement within 3–5 business days. "
            "Is there anything else I can help you with?"
        )
    },
)

print("Trajectory 1 defined: billing question misclassified as 'general'")

Trajectory 1 defined: billing question misclassified as 'general'


In [4]:
# ────────────────────────────────────────────────────────────────────────────
# Trajectory 2: Responder is to blame
#
# The classifier got it right (correctly tagged as "technical").
# But the responder gave a cold, jargon-heavy reply with no empathy.
# The annotation scores it low and explains why.
#
# Credit assignment target: RESPONDER
# ────────────────────────────────────────────────────────────────────────────
trajectory_2 = (
    [
        {"role": "user",      "content": "My app keeps crashing whenever I try to upload a photo."},
        # Classifier nailed it — correct category
        {"role": "assistant", "content": "[Category: technical] Processing technical issue."},
        # Responder's output — technically correct steps but robotic and unhelpful tone
        {"role": "assistant", "content": (
            "Error code analysis: image upload module failure detected. "
            "Resolution steps: 1) Clear cache. 2) Reinstall app. 3) Check OS version compatibility. "
            "Escalate to L2 if unresolved."
        )},
        # User signals frustration with the tone/style
        {"role": "user",      "content": "Okay... that's a lot of jargon. Can you explain in plain English?"},
    ],
    # Score + comment annotation: classifier was fine, responder needs warmth and clarity
    {
        "score": 0.3,
        "comment": (
            "The classifier correctly identified this as a technical issue. "
            "However, the responder's reply is full of internal jargon (e.g., 'L2 escalate') "
            "and shows zero empathy. Responses should be written for non-technical users "
            "and acknowledge the frustration before jumping to steps."
        ),
    },
)

print("Trajectory 2 defined: correct classification, but robotic/jargon-heavy response")

Trajectory 2 defined: correct classification, but robotic/jargon-heavy response


In [5]:
# ────────────────────────────────────────────────────────────────────────────
# Trajectory 3: Happy path — both prompts worked well
#
# High score, positive comment. This is important training signal too:
# it tells the optimizer what "good" looks like so it doesn't accidentally
# optimise away things that are already working.
#
# Credit assignment target: Neither (both get reinforcement)
# ────────────────────────────────────────────────────────────────────────────
trajectory_3 = (
    [
        {"role": "user",      "content": "How do I change the email address on my account?"},
        # Classifier correct: general account question
        {"role": "assistant", "content": "[Category: general] Routing to account support."},
        # Responder gave a warm, clear, step-by-step answer
        {"role": "assistant", "content": (
            "Happy to help! To update your email address:\n"
            "1. Go to Account Settings → Profile\n"
            "2. Click 'Edit' next to your email\n"
            "3. Enter the new address and click Save\n\n"
            "You'll receive a verification link at your new address. Let me know if you run into any issues!"
        )},
        {"role": "user",      "content": "Perfect, that worked. Thanks!"},
    ],
    # High score — this is a success example, reinforces current good behaviours
    {"score": 0.98, "comment": "Correct classification and a friendly, clear, step-by-step response."},
)

print("Trajectory 3 defined: happy path — both stages worked correctly")

Trajectory 3 defined: happy path — both stages worked correctly


In [6]:
# Bundle all trajectories together
trajectories = [trajectory_1, trajectory_2, trajectory_3]
print(f"Total trajectories: {len(trajectories)}")
print("  - Trajectory 1: classifier failure (billing misclassified)")
print("  - Trajectory 2: responder failure (cold jargon-heavy reply)")
print("  - Trajectory 3: success (both stages worked)")

Total trajectories: 3
  - Trajectory 1: classifier failure (billing misclassified)
  - Trajectory 2: responder failure (cold jargon-heavy reply)
  - Trajectory 3: success (both stages worked)


---

## Step 3: Run the Optimizer

The API is almost identical to `create_prompt_optimizer`. The only differences are:

- Use `create_multi_prompt_optimizer` instead
- Pass `"prompts"` (plural, a list) instead of `"prompt"` (singular, a string)
- You get back a **list of updated prompt dicts**

### Which strategy to use?

The same three strategies (`metaprompt`, `gradient`, `prompt_memory`) are available. For multi-prompt pipelines, `gradient` is especially powerful — its step-by-step failure tracing is exactly what you need to correctly assign blame across agents.

We'll use `gradient` here to get the sharpest credit assignment.

In [7]:
optimizer = create_multi_prompt_optimizer(
    "anthropic:claude-haiku-4-5-20251001",
    kind="gradient",               # Best for credit assignment across agents
    config={"max_reflection_steps": 2},
)

# Notice: key is "prompts" (plural), value is our list of prompt dicts
updated_prompts = optimizer.invoke({
    "trajectories": trajectories,
    "prompts": prompts,
})

print("Optimization complete. Updated prompts:")
for p in updated_prompts:
    print(f"\n[{p['name']}]")
    print(p['prompt'])

Optimization complete. Updated prompts:

[classifier]

You are a customer support assistant. For each message:

1. **Classify** the message into one of these categories:
   - **Billing**: Charges, refunds, subscriptions, payment issues
   - **Technical**: App crashes, bugs, performance issues, broken features
   - **Account**: Password resets, email changes, account settings, profile updates
   - **General**: Other inquiries

2. **Respond** with empathy and clarity:
   - Start by acknowledging the customer's specific issue and any frustration
   - Use plain language—avoid internal jargon, acronyms (like "L2"), or technical terms unless the customer used them first
   - Provide clear, step-by-step solutions when applicable
   - If you cannot fully resolve the issue, explain next steps or escalation clearly in customer-friendly terms
   - End by confirming they're satisfied or offering further help

Format: [Category: X] followed by your empathetic, helpful response.


[responder]
You ar

---

## Step 4: Analyse the Results — Credit Assignment in Action

Let's compare what changed and what didn't. This is where you see credit assignment pay off.

In [8]:
# Build a lookup for easy comparison
original = {p["name"]: p["prompt"] for p in prompts}
updated  = {p["name"]: p["prompt"] for p in updated_prompts}

print("=" * 70)
print("BEFORE vs AFTER — Side-by-Side Comparison")
print("=" * 70)

for name in original:
    before = original[name]
    after  = updated[name]
    changed = before != after

    print(f"\n{'▶' if changed else '○'} [{name.upper()}] {'← CHANGED' if changed else '← unchanged'}")
    print(f"\n  BEFORE: {before}")
    print(f"\n  AFTER:  {after}")
    print("-" * 70)

BEFORE vs AFTER — Side-by-Side Comparison

▶ [CLASSIFIER] ← CHANGED

  BEFORE: You classify customer support messages into categories.

  AFTER:  
You are a customer support assistant. For each message:

1. **Classify** the message into one of these categories:
   - **Billing**: Charges, refunds, subscriptions, payment issues
   - **Technical**: App crashes, bugs, performance issues, broken features
   - **Account**: Password resets, email changes, account settings, profile updates
   - **General**: Other inquiries

2. **Respond** with empathy and clarity:
   - Start by acknowledging the customer's specific issue and any frustration
   - Use plain language—avoid internal jargon, acronyms (like "L2"), or technical terms unless the customer used them first
   - Provide clear, step-by-step solutions when applicable
   - If you cannot fully resolve the issue, explain next steps or escalation clearly in customer-friendly terms
   - End by confirming they're satisfied or offering further hel

### What you should see

**`classifier` prompt** — should be substantially updated. The optimizer traces trajectory 1's failure back to the misclassification step. Expected improvements:
- Explicit list of valid categories (`billing`, `technical`, `general`)
- Examples or keywords to guide accurate categorisation
- Instructions for ambiguous edge cases

**`responder` prompt** — should also be updated, but for different reasons (trajectory 2). Expected improvements:
- Instruction to write in plain, non-jargon language
- Instruction to acknowledge user frustration before jumping to steps
- Tone guidance (empathetic, warm)

The key insight: **each prompt gets updated for its own failures**, not for what the other stage did wrong. The responder prompt won't get "fix billing classification" added to it — that's the classifier's job.

---

## Why This Matters More Than It Seems

Consider the alternative: running `create_prompt_optimizer` separately on each prompt with the same trajectories. What would happen?

- The **classifier** optimizer sees the full conversation including "robotic tone" feedback — it might try to add tone instructions to the classification step (wrong place)
- The **responder** optimizer sees the misclassification failure — it might try to "fix" its own prompt to handle bad input rather than expecting clean input from upstream

`create_multi_prompt_optimizer` avoids this cross-contamination by reasoning about the pipeline as a whole.

---

## Bonus: Iterative Optimization (The Real-World Pattern)

In production you don't run the optimizer once. You run it in a loop — each night, or after every N conversations:

```python
# ─── Nightly optimization loop (pseudocode) ─────────────────────────────
# prompts starts as the initial definitions above.

for batch in nightly_conversation_batches:
    trajectories = collect_and_annotate(batch)     # pull from your eval pipeline
    updated_prompts = optimizer.invoke({
        "trajectories": trajectories,
        "prompts": prompts,                         # feed in the CURRENT best prompts
    })
    prompts = updated_prompts                       # update for next iteration
    deploy(prompts)                                  # push to production
```

Each iteration compounds: the optimizer builds on the already-improved prompts from the previous round.

**Important:** Always pass the *current best* prompts as input (not the originals). This is how improvements accumulate over time rather than starting from scratch each night.

---

## Summary

```python
from langmem import create_multi_prompt_optimizer

# 1. Define named prompts for each stage of your pipeline
prompts = [
    {"name": "stage_a", "prompt": "You do X"},
    {"name": "stage_b", "prompt": "You do Y"},
]

# 2. Collect trajectories — same format as single-prompt optimizer
trajectories = [(messages, annotation), ...]

# 3. Create optimizer — same strategies available (gradient recommended for pipelines)
optimizer = create_multi_prompt_optimizer(
    "anthropic:claude-sonnet-4-6",
    kind="gradient",
    config={"max_reflection_steps": 3},
)

# 4. Invoke — note "prompts" (plural) as the key
updated_prompts = optimizer.invoke({
    "trajectories": trajectories,
    "prompts": prompts,        # ← plural key, list of dicts
})

# 5. Result: list of updated prompt dicts, same structure as input
for p in updated_prompts:
    print(p["name"], "→", p["prompt"])
```

**Key takeaways:**
- Use `create_multi_prompt_optimizer` whenever you have ≥2 LLM stages in your pipeline
- **Credit assignment** is the main value: the optimizer blames the right stage, not all stages
- The `gradient` strategy works best for pipelines — it traces failures back through the call chain
- Always pass the *current* prompts (not the originals) when running iteratively
- Mix annotation types: `revised` is ideal for classification errors, `score+comment` for quality issues, `None` for implicit signal from user pushback